In [ ]:
import torch
import torchaudio
import librosa
import soundfile as sf
import random
import io
from pydub import AudioSegment

# Extra
from pydub.generators import WhiteNoise

c:\Users\mxlov\anaconda3\envs\speech-recog-go\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


## Variable

In [2]:
AUDIO_PATH = './speech.wav'
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

## Transcriber

In [3]:
def Transcriber (audiopath):
    waveform, sample_rate = torchaudio.load(audiopath)
    if sample_rate != bundle.sample_rate:
        waveform = torchaudio.functional.resample(waveform, sample_rate, bundle.sample_rate)

    with torch.inference_mode():
        emissions, _ = model(waveform)

    class GreedyCTCDecoder(torch.nn.Module):
        def __init__(self, labels):
            super().__init__()
            self.labels = labels
            self.blank = 0

        def forward(self, emissions: torch.Tensor):
            idx = torch.argmax(emissions, dim=-1)
            idx = torch.unique_consecutive(idx, dim=-1)
            idx = [i for i in idx if i != self.blank]
            return " ".join([self.labels[i] for i in idx])

    decoder = GreedyCTCDecoder(bundle.get_labels())
    txt = decoder(emissions[0])
    txt = txt.replace("|", " ")
    return txt
        

## Support Function

In [4]:
def ExportMenu (audio):
    filename = None

    while True:
        print("Input File name, must end with .wav:")
        filename = input()

        if not filename.endswith('.wav'):
            print("Filename must end with .wav")
        else:
            break

    audio.export(filename, format='wav')
    print("File Save Success")


In [ ]:
def Menu ():
    audio = AudioSegment.from_wav(AUDIO_PATH)

    while True:
        print("1. Adjust Volume")
        print("2. Adjust Pitch")
        print("3. Insert Fade In")
        print("4. Insert Fade Out")
        print("5. Back to Main Menu")
        print("11. Extra White Noise")
        print("22. Reverse Audio")
        print(">> ")
        cc = input()

        if cc == "1":
            while True:
                print("Input Volume: ")
                db = int(input())

                if db >= -10000 and db <= 100:
                    break
                else:
                    print("Invalid Range Input")

            audio = audio + db
            ExportMenu(audio)

        elif cc == "2":
            while True:
                print("Input Pitch Rate: ")
                pitch = int(input())

                if pitch >= -10 and pitch <= 10:
                    break
                else:
                    print("Invalid Range Input")

            shifted, sample_rate = librosa.load(AUDIO_PATH)
            shifted = librosa.effects.pitch_shift(shifted, sr=sample_rate, n_steps=pitch)

            buffer = io.BytesIO()
            sf.write(buffer, shifted, samplerate=sample_rate, format='WAV')
            buffer.seek(0)

            audio = AudioSegment.from_file(buffer, format='WAV')
            ExportMenu(audio)

        elif cc == "3":
            dur = random.randint(1, 5) * 1000
            audio = audio.fade_in(dur)
            ExportMenu(audio)

        elif cc == "4":
            dur = random.randint(1, 5) * 1000
            audio = audio.fade_out(dur)
            ExportMenu(audio)

        elif cc == "5":
            break

        elif cc == "11":
            noice = WhiteNoise().to_audio_segment(duration=len(audio))
            noice = noice - 30
            audio = audio.overlay(noice)
            ExportMenu(audio)

        elif cc == "22":
            audio.reverse()
            ExportMenu(audio)
        
        else:
            print("Invalid Input")

def MainMenu ():
    while True:
        print("1. Audio Augmentation")
        print("2. Predict (Transcribe) Text")
        print("3. Exit")
        print(">> ")
        cc = input()

        if cc == "1":
            Menu()
        elif cc == "2":
            txt = Transcriber(AUDIO_PATH)
            print(f"Transcribe Result:\n {txt}")
        elif cc == "3":
            break
        else:
            print("Invalid Input")
    

In [ ]:
MainMenu()